# How the retargeting works

`smpl_to_carla.py` turns an SMPL motion into the `.pkl` files the benchmark animates its
pedestrians from. This notebook opens that up a step at a time, so you can see what each stage
does before trusting the command-line tool with a whole dataset.

It follows Sec. 3.3 of the paper. Each section names the equation it implements and calls the
same function the tool uses, so nothing here is a reimplementation that can drift out of step
with the code you actually run.

Nothing below needs CARLA running.

## The problem

| | SMPL | CARLA (Unreal) |
|---|---|---|
| joints | 24 (first 22 used) | 26 bones |
| rotation | axis-angle, per joint | Euler angles, per bone |
| handedness | right-handed | left-handed |
| up axis | +Y | +Z |
| rest pose | T-pose | a deformed, non-T rest pose |

The last row causes most of the work. CARLA's walker rig is not in a T-pose at rest, so SMPL's
rotations cannot be handed over directly -- each one has to be re-expressed relative to the pose
the rig is actually in.

In [ ]:
import sys
import numpy as np

sys.path.insert(0, '.')          # run this notebook from tools/motion/

import smpl_to_carla as s2c

np.set_printoptions(precision=4, suppress=True)
print(f'{len(s2c.CARLA_BONES)} CARLA bones, {len(s2c.SMPL_JOINTS)} SMPL joints')
print(f'{len(s2c.JOINT_MAP)} driven by SMPL, '
      f'{len(s2c.CARLA_BONES) - len(s2c.JOINT_MAP)} left at rest')

## A motion to work with

Point `SMPL_FILE` at any SMPL `.npz` -- an AMASS sequence, a HumanML3D clip you have built, or
the output of a video lifting pipeline. See `README.md` for where to get each.

If the file is missing, the cell falls back to a synthetic walk generated below, so the notebook
runs out of the box. The synthetic motion is deliberately crude -- sinusoidal hips, knees,
shoulders and elbows -- but it exercises every step of the conversion.

In [ ]:
SMPL_FILE = 'your_motion.npz'      # <- point this at a real SMPL sequence


def synthetic_walk(frames=120, fps=20.0):
    """A simple sinusoidal gait, for when no SMPL file is to hand.

    Joints swing about more than one axis. Real ones do, and a single-axis
    motion would make the rotation-map comparison further down vacuous: for a
    rotation about one axis the exponential map and the XYZ Euler reading
    coincide exactly.
    """
    phase = np.linspace(0, 4 * np.pi, frames)
    omega = np.zeros((frames, 22, 3))

    def swing(joint, amplitude, offset=0.0, axis=0):
        omega[:, s2c.SMPL_JOINTS.index(joint), axis] += amplitude * np.sin(phase + offset)

    # flexion, the dominant component
    swing('L_Hip', 0.45); swing('R_Hip', 0.45, np.pi)
    swing('L_Knee', -0.5, -np.pi / 2); swing('R_Knee', -0.5, np.pi / 2)
    swing('L_Shoulder', 0.35, np.pi); swing('R_Shoulder', 0.35)
    swing('L_Elbow', -0.2, np.pi); swing('R_Elbow', -0.2)

    # secondary rotation and abduction
    swing('L_Hip', 0.10, np.pi / 3, axis=1); swing('R_Hip', -0.10, np.pi / 3, axis=1)
    swing('L_Shoulder', 0.18, np.pi, axis=2); swing('R_Shoulder', -0.18, 0.0, axis=2)
    swing('Spine1', 0.06, 0.0, axis=1)

    trans = np.zeros((frames, 3))
    trans[:, 2] = np.linspace(0, frames / fps * 1.3, frames)   # forward at 1.3 m/s
    trans[:, 1] = 0.02 * np.sin(2 * phase)                     # slight vertical bob
    return omega, trans, fps


try:
    omega, trans, fps = s2c.read_smpl(SMPL_FILE)
    source = SMPL_FILE
except (FileNotFoundError, OSError):
    omega, trans, fps = synthetic_walk()
    source = 'synthetic walk'

T = len(omega)
print(f'{source}: {T} frames' + (f' at {fps:g} fps' if fps else ''))
print(f'axis-angle {omega.shape}, translation {trans.shape}')

magnitude = np.degrees(np.linalg.norm(omega, axis=-1))
print(f'joint rotation: median {np.median(magnitude):.1f} deg, max {magnitude.max():.1f} deg')

## Eq. 3 -- axis-angle to rotation matrix

$$R_{t,j} = \exp(\hat{\omega}_{t,j})$$

SMPL stores each joint as a rotation vector: direction is the axis, magnitude is the angle in
radians. Rodrigues' formula turns that into a matrix. Everything after this is a change of basis,
and those compose as matrix products but not as raw angle triples -- which is the whole reason
for moving into matrices here.

In [ ]:
R = s2c.axis_angle_to_matrix(omega)

print('rotation matrices:', R.shape)
print('proper rotations  :', np.allclose(np.linalg.det(R), 1.0))
print('orthonormal       :', np.allclose(R @ R.swapaxes(-1, -2), np.eye(3), atol=1e-9))

## Eq. 4 -- into CARLA's basis

$$R^{\text{CARLA}}_{t,j} = C\,R_{t,j}\,C^{-1}$$

A rotation only means anything relative to a basis, so moving between conventions is a similarity
transform. `BASIS` carries SMPL's axes onto CARLA's.

Conjugation alone cannot change handedness, so the chirality flip is applied to the rotation
vector instead, by negating its X component (`MIRROR`). Doing it there keeps every matrix in the
pipeline a proper rotation, which is what CARLA's bone API expects.

In [ ]:
for name, axis in [('SMPL +X', [1, 0, 0]),
                   ('SMPL +Y (up)', [0, 1, 0]),
                   ('SMPL +Z (forward)', [0, 0, 1])]:
    print(f'{name:20s} -> {s2c.BASIS @ np.array(axis, dtype=float)}')

print('\nchirality flip applied to the rotation vector:', s2c.MIRROR)

## Eq. 5 -- joint correspondence and the spine

$$R^{\text{CARLA}}_{t,\text{Spine1}} = R^{\text{CARLA}}_{t,\text{Spine2}} \cdot R^{\text{CARLA}}_{t,\text{Spine3}}$$

21 CARLA bones take a SMPL joint directly. SMPL has three spine joints where CARLA has two, so
two of them fold into one bone, which then carries the whole bend of that stretch of back.

In the code this reads `Spine3 @ Spine2`, which looks reversed but is not. The pipeline follows
Unreal's **row-vector** convention: transforms act on row vectors from the right, so products
read left to right in the order the rotations apply. The same convention governs the forward
kinematics, where bones compose as `child @ parent` and the 4x4 keeps its translation in the last
*row*. Read these products in the usual column-vector order and every composition in the pipeline
looks backwards.

In [ ]:
for bone, joint in s2c.JOINT_MAP[:6]:
    print(f'  {bone:18s} <- {joint}')
print('  ...')

driven = {bone for bone, _ in s2c.JOINT_MAP}
print('\nleft at rest:', [b for b in s2c.CARLA_BONES if b not in driven])

## Eq. 6 -- referencing the rig's rest pose

$$\Delta R_{t,j} = (R^{\text{ref}}_j)^{-1} R^{\text{CARLA}}_{t,j} R^{\text{ref}}_j$$

CARLA wants each bone's rotation relative to its parent, measured from the rig's own rest pose.
Getting $R^{\text{ref}}$ means walking the bone tree and composing each bone's local transform
with its parent's -- ordinary forward kinematics.

Two conventions in CARLA's dump to watch: lengths are centimetres, and the stored angles are
negated relative to the right-handed frame used here. `load_rest_pose` handles both.

In [ ]:
rest_loc, rest_rot, tree = s2c.load_rest_pose('male')
_, rest_abs_rot = s2c.forward_kinematics(rest_loc[None], rest_rot[None], tree)
reference = rest_abs_rot[0] @ s2c.BASIS

rest_points, _ = s2c.forward_kinematics(rest_loc[None], rest_rot[None], tree)

print('rest offsets   ', rest_loc.shape)
print('R_ref          ', reference.shape)
print(f'rig spans      {np.ptp(rest_points[0][:, 2]):.2f} m head to toe')

## Eq. 7 -- back out to Euler angles

$$\phi_{t,j} = \mathrm{Euler}_{XYZ}(\Delta R_{t,j})$$

CARLA's bone API takes angles, so the last step decomposes to XYZ Euler and converts to degrees.
`retarget` performs Eqs. 3 to 7 in one call. The result is `(T, 26, 3)`, ordered
**(roll, pitch, yaw)** -- which is what the runtime assumes when it unpacks the array.

In [ ]:
pose_data = s2c.retarget(omega, rig='male')
print('pose_data', pose_data.shape)

print('\nframe 0:')
for bone in ['crl_root', 'crl_hips__C', 'crl_arm__L', 'crl_thigh__R']:
    print(f'  {bone:16s} {pose_data[0, s2c.CARLA_BONES.index(bone)]}')

### The rotation map

`retarget` takes a `rotation` argument. The default, `'euler'`, reads each axis-angle triple as
three sequential XYZ rotations, which is what produced the 4,730 released motions. Passing
`'expmap'` applies Eq. 3 literally.

The two agree to first order and drift apart as angles grow. On pedestrian motion that comes to
roughly a degree per joint, concentrated wherever the rotations are largest -- usually the
shoulders. Neither is measurably closer to the SMPL body once retargeted, so the reason to keep
the default is consistency with the released set, not accuracy.

In [ ]:
a = s2c.axis_angle_to_matrix(omega * s2c.MIRROR)
b = s2c.euler_xyz_to_matrix(omega * s2c.MIRROR)
relative = np.einsum('tjik,tjlk->tjil', a, b)
gap = np.degrees(np.arccos(np.clip((np.trace(relative, axis1=2, axis2=3) - 1) / 2, -1, 1)))

print(f'expmap vs euler: mean {gap.mean():.2f} deg, '
      f'95th pct {np.percentile(gap, 95):.2f} deg, max {gap.max():.2f} deg')

## Seeing it

Rotations are hard to check by eye, so run them back through the forward kinematics -- this time
with the animated rotations rather than the rest ones -- and draw the skeleton.

The rig comes out of the FK with **-Z** up, which is why the vertical axis is negated below.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

relative_rot = s2c.euler_xyz_to_matrix(np.deg2rad(-pose_data))
points, _ = s2c.forward_kinematics(
    np.broadcast_to(rest_loc, (T, s2c.N_BONES, 3)).copy(), relative_rot, tree)


def bone_edges(tree):
    """Flatten the bone hierarchy into (child, parent) index pairs."""
    edges = []

    def walk(node, parent):
        name, children = list(node.items())[0]
        i = s2c.CARLA_BONES.index(name)
        if parent is not None:
            edges.append((i, parent))
        for child in (children or []):
            walk(child, i)

    walk(tree[0], None)
    return edges


EDGES = bone_edges(tree)
view = points - points[:, :1]
view = np.stack([view[..., 0], view[..., 1], -view[..., 2]], -1)    # -Z is up
print(f'{len(EDGES)} bones, figure spans {np.ptp(view[..., 2]):.2f} m vertically')

In [ ]:
fig = plt.figure(figsize=(4.8, 5.4))
ax = fig.add_subplot(projection='3d')


def draw(frame):
    ax.clear()
    p = view[frame]
    for a, b in EDGES:
        ax.plot(*zip(p[a], p[b]), color='#d62728', lw=2.2, solid_capstyle='round')
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], s=7, c='0.2', depthshade=False)
    ax.set_xlim(-0.55, 0.55); ax.set_ylim(-0.55, 0.55); ax.set_zlim(-1.05, 0.75)
    ax.set_box_aspect((1, 1, 1.6)); ax.set_axis_off()
    ax.view_init(elev=10, azim=-72)
    ax.set_title(f'frame {frame}', fontsize=11)


anim = FuncAnimation(fig, draw, frames=range(0, T, 2), interval=100)
plt.close(fig)
HTML(anim.to_jshtml(default_mode='loop'))

## Writing the file

The runtime loads a joblib pickle with two keys: `pose_data` `(T, 26, 3)` float64 degrees, and
`transl` `(1, T, 3)` float32 metres. The leading axis on `transl` is a batch dimension the loader
squeezes away.

`transl` is stored relative to its own first frame. The same motion gets reused at many spawn
points, so the runtime adds the spawn location, the heading and the rig's hip height at playback.

`convert()` does the read, resample, retarget and recentre in one call -- it is exactly what the
command-line tool runs per file.

In [ ]:
import joblib

motion = {'pose_data': pose_data.astype(np.float64),
          'transl': (trans - trans[0]).astype(np.float32)[None]}

joblib.dump(motion, 'example_motion.pkl')
check = joblib.load('example_motion.pkl')
print('pose_data', check['pose_data'].shape, check['pose_data'].dtype)
print('transl   ', check['transl'].shape, check['transl'].dtype)

## Checking against the released motions

The five bones with no SMPL counterpart never move, whatever the input, so they always hold the
rig's rest rotation. That makes them a fingerprint: if the reference handling is right they match
the released files exactly. It also identifies which rig a set of motions was built against --
the released ones use `male`.

Needs the motion data downloaded (`scripts/download_motion_data.sh`); skipped otherwise.

In [ ]:
import os

released_file = os.path.join('..', '..', 'data', 'motions', '000007.pkl')

if os.path.exists(released_file):
    released = joblib.load(released_file)['pose_data'][0]
    at_rest = s2c.retarget(np.zeros((1, 22, 3)), rig='male')[0]
    for bone in ['crl_root', 'crl_eye__L', 'crl_toeEnd__R', 'crl_toeEnd__L']:
        i = s2c.CARLA_BONES.index(bone)
        print(f'  {bone:15s} {np.round(at_rest[i], 3)}  '
              f'released {np.round(released[i], 3)}  '
              f'{np.allclose(at_rest[i], released[i], atol=1e-3)}')
else:
    print(f'{released_file} not found -- run scripts/download_motion_data.sh to compare')

---

## Using a converted motion

Put the `.pkl` in `data/motions/` and add its file ID to one of the behaviour CSVs in
`data/csvs/`. The category decides how the benchmark uses it: motions listed in `Crossing.csv` or
`Attempting.csv` trigger when the ego vehicle closes to the activation distance, while
`Not_Crossing.csv` entries play as ambient idle behaviour.

For batch conversion use the tool directly:

```bash
python3 tools/motion/smpl_to_carla.py /path/to/smpl/ -o data/motions/
```